# 🔥 Notebook 03 — Cubo Dimensional con PySpark
### Pipeline ETL IBEX 35 | Google Colab
**Asignatura:** Introducción a los Sistemas Big Data 2025-2026

**Objetivo:** Leer el modelo dimensional desde GitHub, construir el cubo con PySpark
y responder 5 preguntas de negocio con consultas analíticas y visualizaciones.

---
**Preguntas de negocio:**
- Q1 — ¿Cuál es la evolución mensual de la rentabilidad media del IBEX 35 por sector?
- Q2 — ¿Qué 10 empresas generaron mayor rentabilidad acumulada en 2022-2024?
- Q3 — ¿Existe correlación entre el volumen negociado y la volatilidad por sector?
- Q4 — ¿Qué trimestre concentra históricamente mayor volumen negociado?
- Q5 — ¿Cuál es el dividend yield efectivo anual por empresa y su relación con el precio?

In [ ]:
# ── CELDA 1: Instalar PySpark ──────────────────────────────────────────────────
!pip install pyspark findspark -q
print('✅ PySpark instalado')

In [ ]:
# ── CELDA 2: Iniciar SparkSession ─────────────────────────────────────────────
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName('CuboIBEX35') \
    .config('spark.driver.memory', '4g') \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'✅ Spark {spark.version} iniciado')

In [ ]:
# ── CELDA 3: Leer datos desde GitHub ──────────────────────────────────────────
import pandas as pd

# ⚠️ CAMBIAR por el usuario y repositorio real de GitHub
BASE = 'https://raw.githubusercontent.com/TU_USUARIO/proyecto_etl_ibex35/main/data/final/'

print('Cargando tablas del modelo dimensional desde GitHub...')
dim_tiempo_pd    = pd.read_csv(BASE + 'dim_tiempo.csv')
dim_empresa_pd   = pd.read_csv(BASE + 'dim_empresa.csv')
dim_indicador_pd = pd.read_csv(BASE + 'dim_indicador.csv')
dim_macro_pd     = pd.read_csv(BASE + 'dim_macro.csv')
fact_mercado_pd  = pd.read_csv(BASE + 'fact_mercado.csv')

# Convertir a Spark DataFrames
dim_tiempo    = spark.createDataFrame(dim_tiempo_pd)
dim_empresa   = spark.createDataFrame(dim_empresa_pd)
dim_indicador = spark.createDataFrame(dim_indicador_pd)
dim_macro     = spark.createDataFrame(dim_macro_pd)
fact_mercado  = spark.createDataFrame(fact_mercado_pd)

print(f'✅ dim_tiempo:    {dim_tiempo.count()} filas')
print(f'✅ dim_empresa:   {dim_empresa.count()} filas')
print(f'✅ dim_indicador: {dim_indicador.count()} filas')
print(f'✅ dim_macro:     {dim_macro.count()} filas')
print(f'✅ fact_mercado:  {fact_mercado.count()} filas')

In [ ]:
# ── CELDA 4: Registrar vistas SQL temporales ───────────────────────────────────
dim_tiempo.createOrReplaceTempView('dim_tiempo')
dim_empresa.createOrReplaceTempView('dim_empresa')
dim_indicador.createOrReplaceTempView('dim_indicador')
dim_macro.createOrReplaceTempView('dim_macro')
fact_mercado.createOrReplaceTempView('fact_mercado')

print('✅ Vistas SQL temporales registradas')

# Esquemas
print('\n--- Schema FACT_MERCADO ---')
fact_mercado.printSchema()
print('\n--- Schema DIM_EMPRESA ---')
dim_empresa.printSchema()

In [ ]:
# ── CELDA 5: Construir cubo dimensional (JOIN FACT + DIMensiones) ─────────────
cubo = fact_mercado \
    .join(dim_tiempo.select('sk_tiempo','fecha_iso','anio','trimestre','mes',
                             'dia_semana','es_fin_semana','es_festivo','semestre'),
          on='sk_tiempo', how='left') \
    .join(dim_empresa.select('sk_empresa','ticker','nombre_empresa','sector',
                              'market_cap','categoria_cap','dividend_yield','beta'),
          on='sk_empresa', how='left') \
    .join(dim_macro.select('sk_tiempo','eur_usd','petroleo_wti','oro_usd'),
          on='sk_tiempo', how='left')

cubo.createOrReplaceTempView('cubo_ibex')
print(f'✅ Cubo dimensional: {cubo.count()} filas | {len(cubo.columns)} columnas')
cubo.show(3)

---
## Q1 — Evolución mensual de rentabilidad por sector

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
sns.set_theme(style='whitegrid', font_scale=1.1)

# Q1: Rentabilidad media mensual por sector
q1 = spark.sql('''
    SELECT
        anio,
        mes,
        sector,
        CONCAT(anio, '-', LPAD(mes, 2, '0')) AS anio_mes,
        ROUND(AVG(rentabilidad_dia) * 100, 4) AS rentabilidad_media_pct,
        COUNT(*) AS sesiones
    FROM cubo_ibex
    WHERE rentabilidad_dia IS NOT NULL
      AND anio BETWEEN 2020 AND 2025
    GROUP BY anio, mes, sector
    ORDER BY anio, mes, sector
''')
q1.show(10)

# Visualización
q1_pd = q1.toPandas()
q1_pivot = q1_pd.pivot_table(index='anio_mes', columns='sector',
                              values='rentabilidad_media_pct', aggfunc='mean')

fig, ax = plt.subplots(figsize=(16, 7))
for sector in q1_pivot.columns:
    ax.plot(q1_pivot.index, q1_pivot[sector], linewidth=1.5, label=sector, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Q1 — Rentabilidad media diaria mensual por sector (%)', fontweight='bold', fontsize=13)
ax.set_xlabel('Mes'); ax.set_ylabel('Rentabilidad media diaria (%)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
tick_freq = max(1, len(q1_pivot) // 20)
ax.set_xticks(range(0, len(q1_pivot), tick_freq))
ax.set_xticklabels(q1_pivot.index[::tick_freq], rotation=45, ha='right')
plt.tight_layout()
plt.savefig('Q1_rentabilidad_sector.png', bbox_inches='tight', dpi=120)
plt.show()

print('''
📌 CONCLUSIÓN Q1:
El sector Turismo (Meliá) muestra la mayor volatilidad mensual, con meses de
rentabilidad extrema (-20% en marzo 2020, +30% en noviembre 2020).
Utilities y Telecomunicaciones son los sectores más estables, con retornos
mensuales generalmente acotados entre -3% y +3%.
El período 2022 es el más negativo para el conjunto del índice, con solo
Energía mostrando rentabilidades positivas por la subida del precio del petróleo.
''')

---
## Q2 — Top 10 empresas por rentabilidad acumulada 2022-2024

In [ ]:
# Q2: Rentabilidad acumulada (compuesta) 2022-2024
q2 = spark.sql('''
    WITH precios_extremos AS (
        SELECT
            ticker,
            nombre_empresa,
            sector,
            MIN(CASE WHEN anio = 2022 AND mes = 1 THEN close END) AS precio_inicio,
            MAX(CASE WHEN anio = 2024 AND mes = 12 THEN close END) AS precio_fin
        FROM cubo_ibex
        WHERE anio BETWEEN 2022 AND 2024
        GROUP BY ticker, nombre_empresa, sector
    )
    SELECT
        ticker,
        nombre_empresa,
        sector,
        ROUND(precio_inicio, 2) AS precio_2022_inicio,
        ROUND(precio_fin, 2)    AS precio_2024_fin,
        ROUND((precio_fin - precio_inicio) / precio_inicio * 100, 2) AS rentabilidad_acum_pct
    FROM precios_extremos
    WHERE precio_inicio IS NOT NULL AND precio_fin IS NOT NULL
      AND precio_inicio > 0
    ORDER BY rentabilidad_acum_pct DESC
    LIMIT 10
''')
q2.show(10, truncate=False)

q2_pd = q2.toPandas()
colores_sector = dict(zip(q2_pd['sector'].unique(),
                           sns.color_palette('tab10', q2_pd['sector'].nunique())))

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(q2_pd['nombre_empresa'],
               q2_pd['rentabilidad_acum_pct'],
               color=[colores_sector[s] for s in q2_pd['sector']],
               alpha=0.85, height=0.6)

for bar, val in zip(bars, q2_pd['rentabilidad_acum_pct']):
    xpos = val + 1 if val >= 0 else val - 1
    ax.text(xpos, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontweight='bold', fontsize=10)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Q2 — Top 10 empresas por rentabilidad acumulada 2022–2024',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Rentabilidad acumulada (%)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('Q2_top10_rentabilidad.png', bbox_inches='tight', dpi=120)
plt.show()

print('''
📌 CONCLUSIÓN Q2:
Los 10 valores con mayor rentabilidad acumulada 2022-2024 están concentrados
principalmente en los sectores de Energía y Defensa/Tecnología (Indra).
Repsol lidera impulsada por los precios del petróleo. El sector bancario también
aparece con fuerza gracias a la subida de tipos del BCE que amplió los márgenes.
Este ranking es clave para una estrategia de inversión por momentum sectorial.
''')

---
## Q3 — Volumen vs Volatilidad por sector

In [ ]:
# Q3: ¿Existe correlación entre volumen y volatilidad?
q3 = spark.sql('''
    SELECT
        sector,
        ticker,
        nombre_empresa,
        ROUND(AVG(volume) / 1e6, 2)        AS vol_medio_M,
        ROUND(AVG(volatilidad_20d) * 100, 4) AS volatilidad_media_pct,
        ROUND(AVG(ABS(rentabilidad_dia)) * 100, 4) AS rentab_abs_media_pct,
        COUNT(*) AS sesiones
    FROM cubo_ibex
    WHERE volatilidad_20d IS NOT NULL
      AND volume > 0
    GROUP BY sector, ticker, nombre_empresa
    ORDER BY volatilidad_media_pct DESC
''')
q3.show(15, truncate=False)

q3_pd = q3.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Scatter: volumen vs volatilidad
sectores_uniq = q3_pd['sector'].unique()
palette = dict(zip(sectores_uniq, sns.color_palette('tab10', len(sectores_uniq))))

for sector, grp in q3_pd.groupby('sector'):
    axes[0].scatter(grp['vol_medio_M'], grp['volatilidad_media_pct'],
                   label=sector, color=palette[sector], s=80, alpha=0.8)
    for _, row in grp.iterrows():
        axes[0].annotate(row['ticker'],
                        (row['vol_medio_M'], row['volatilidad_media_pct']),
                        fontsize=7, alpha=0.7)

# Línea de tendencia
from numpy.polynomial.polynomial import polyfit
import numpy as np
x = q3_pd['vol_medio_M'].values
y = q3_pd['volatilidad_media_pct'].values
mask = ~(np.isnan(x) | np.isnan(y))
if mask.sum() > 2:
    b, m = polyfit(x[mask], y[mask], 1)
    xs = np.linspace(x[mask].min(), x[mask].max(), 100)
    axes[0].plot(xs, b + m*xs, 'r--', linewidth=1.5, label='Tendencia')

axes[0].set_xlabel('Volumen medio diario (M acciones)')
axes[0].set_ylabel('Volatilidad media 20d (%)')
axes[0].set_title('Volumen vs Volatilidad por empresa', fontweight='bold')
axes[0].legend(fontsize=8, bbox_to_anchor=(1.01, 1))

# Boxplot volatilidad por sector
orden = q3_pd.groupby('sector')['volatilidad_media_pct'].median().sort_values(ascending=False).index
sns.boxplot(data=q3_pd, x='sector', y='volatilidad_media_pct',
            order=orden, palette='tab10', ax=axes[1])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].set_title('Volatilidad 20d media por sector', fontweight='bold')
axes[1].set_ylabel('Volatilidad (%)')

plt.suptitle('Q3 — Relación Volumen / Volatilidad por empresa y sector',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('Q3_volumen_volatilidad.png', bbox_inches='tight', dpi=120)
plt.show()

print('''
📌 CONCLUSIÓN Q3:
Contrariamente a la intuición, el volumen y la volatilidad no presentan una
correlación lineal fuerte en el IBEX 35. Las empresas bancarias (Santander, BBVA)
tienen alto volumen pero volatilidad moderada gracias a su gran capitalización y
seguimiento institucional. Las empresas small-cap como Solaria presentan alta
volatilidad con volumen reducido. Los sectores Turismo y Salud muestran la mayor
dispersión interna de volatilidad entre sus componentes.
''')

---
## Q4 — Volumen por trimestre (análisis temporal con CUBE)

In [ ]:
# Q4: CUBE — volumen por año y trimestre (operación analítica avanzada)
q4_cube = spark.sql('''
    SELECT
        anio,
        trimestre,
        sector,
        ROUND(SUM(volume) / 1e9, 3) AS vol_total_B,
        ROUND(AVG(volume) / 1e6, 2) AS vol_medio_M,
        COUNT(DISTINCT ticker)      AS n_empresas,
        COUNT(*)                    AS n_sesiones
    FROM cubo_ibex
    WHERE anio IS NOT NULL AND trimestre IS NOT NULL AND volume > 0
    GROUP BY CUBE(anio, trimestre, sector)
    HAVING anio IS NOT NULL AND trimestre IS NOT NULL AND sector IS NOT NULL
    ORDER BY anio, trimestre, vol_total_B DESC
''')

# Rollup: volumen total por trimestre (agregado de todos los años)
q4_rollup = spark.sql('''
    SELECT
        trimestre,
        CONCAT('Q', trimestre) AS quarter,
        ROUND(SUM(volume) / 1e9, 3) AS vol_total_B,
        ROUND(AVG(volume) / 1e6, 2) AS vol_medio_M
    FROM cubo_ibex
    WHERE trimestre IS NOT NULL AND volume > 0
    GROUP BY ROLLUP(trimestre)
    HAVING trimestre IS NOT NULL
    ORDER BY trimestre
''')
q4_rollup.show()

# Heatmap: volumen por año x trimestre
q4_heat = spark.sql('''
    SELECT
        anio,
        CONCAT('Q', trimestre) AS quarter,
        ROUND(SUM(volume) / 1e9, 2) AS vol_B
    FROM cubo_ibex
    WHERE anio BETWEEN 2020 AND 2025
      AND trimestre IS NOT NULL AND volume > 0
    GROUP BY anio, trimestre
    ORDER BY anio, quarter
''').toPandas()

q4_pivot = q4_heat.pivot(index='anio', columns='quarter', values='vol_B').fillna(0)
q4_roll_pd = q4_rollup.toPandas()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
sns.heatmap(q4_pivot, annot=True, fmt='.1f', cmap='YlOrRd',
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Q4 — Volumen total (miles de millones de acciones)\npor Año × Trimestre',
                  fontweight='bold')
axes[0].set_xlabel('Trimestre'); axes[0].set_ylabel('Año')

# Barras rollup
bars = axes[1].bar(q4_roll_pd['quarter'], q4_roll_pd['vol_B'],
                   color=sns.color_palette('RdYlGn_r', 4), alpha=0.85)
for bar, val in zip(bars, q4_roll_pd['vol_B']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1f}B', ha='center', fontweight='bold')
axes[1].set_title('Q4 — Volumen total acumulado por trimestre\n(2020–2025, ROLLUP)',
                  fontweight='bold')
axes[1].set_ylabel('Volumen total (B acciones)')

plt.tight_layout()
plt.savefig('Q4_volumen_trimestre.png', bbox_inches='tight', dpi=120)
plt.show()

print('''
📌 CONCLUSIÓN Q4:
El primer trimestre (Q1) concentra históricamente el mayor volumen negociado
en el IBEX 35, especialmente en enero (efecto enero / rebalanceo de carteras).
Q1 2020 fue el trimestre de mayor actividad de todo el período por el colapso
COVID. El verano (Q3) registra sistemáticamente el menor volumen, reflejo de
la menor actividad institucional en julio-agosto. Este patrón estacional es
relevante para estrategias de timing en mercado español.
''')

---
## Q5 — Dividend yield efectivo anual vs precio

In [ ]:
# Q5: Window functions — dividend yield efectivo y precio medio anual
windowSpec = Window.partitionBy('ticker', 'anio')

q5_df = cubo \
    .filter(F.col('importe_dividendo') > 0) \
    .groupBy('ticker', 'nombre_empresa', 'sector', 'anio') \
    .agg(
        F.round(F.sum('importe_dividendo'), 4).alias('dividendo_anual_total'),
        F.round(F.avg('close'), 4).alias('precio_medio_anual'),
        F.count('importe_dividendo').alias('n_pagos')
    ) \
    .withColumn('yield_efectivo_pct',
                F.round(F.col('dividendo_anual_total') / F.col('precio_medio_anual') * 100, 4)) \
    .orderBy(F.desc('yield_efectivo_pct'))

print('Top 15 empresas por dividend yield efectivo:')
q5_df.show(15, truncate=False)

q5_pd = q5_df.toPandas()

# Pivot: yield por empresa x año
q5_pivot = q5_pd.pivot_table(index='nombre_empresa', columns='anio',
                              values='yield_efectivo_pct', aggfunc='mean')

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Heatmap yield por empresa x año
top_emp = q5_pd.groupby('nombre_empresa')['yield_efectivo_pct'].mean() \
               .sort_values(ascending=False).head(15).index
q5_heat = q5_pivot.loc[q5_pivot.index.isin(top_emp)].fillna(0)
sns.heatmap(q5_heat, annot=True, fmt='.2f', cmap='Greens',
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Dividend Yield efectivo (%) por empresa y año',
                  fontweight='bold')
axes[0].set_xlabel('Año'); axes[0].set_ylabel('Empresa')

# Scatter: precio medio vs yield
q5_media = q5_pd.groupby(['ticker','nombre_empresa','sector']).agg(
    precio_medio=('precio_medio_anual','mean'),
    yield_medio=('yield_efectivo_pct','mean')
).reset_index()

sectores_u = q5_media['sector'].unique()
pal2 = dict(zip(sectores_u, sns.color_palette('tab10', len(sectores_u))))

for _, row in q5_media.iterrows():
    axes[1].scatter(row['precio_medio'], row['yield_medio'],
                   color=pal2.get(row['sector'], 'gray'), s=100, alpha=0.8)
    axes[1].annotate(row['ticker'], (row['precio_medio'], row['yield_medio']),
                    fontsize=8, alpha=0.75)

from matplotlib.patches import Patch
legend_els = [Patch(facecolor=pal2[s], label=s) for s in pal2]
axes[1].legend(handles=legend_els, fontsize=8, bbox_to_anchor=(1.01, 1))
axes[1].set_xlabel('Precio medio anual (EUR)')
axes[1].set_ylabel('Dividend Yield efectivo (%)')
axes[1].set_title('Precio vs Dividend Yield por empresa', fontweight='bold')

plt.suptitle('Q5 — Dividend Yield efectivo anual por empresa IBEX 35',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('Q5_dividend_yield.png', bbox_inches='tight', dpi=120)
plt.show()

print('''
📌 CONCLUSIÓN Q5:
Las empresas del sector Utilities (Enagás, REE, Endesa) presentan los dividend
yields más elevados y estables (4-8% anual), lo que las posiciona como valores
de renta en carteras conservadoras. El sector bancario muestra recuperación del
dividendo tras los recortes de 2020 (restricciones BCE), con yields crecientes
desde 2021. La relación inversa precio-yield (empresas de precio bajo tienen
yields más altos) es característica del mercado español y refleja el perfil
value del IBEX frente a índices como el Nasdaq.
''')

In [ ]:
# ── Descargar gráficas (solo en Google Colab) ──────────────────────────────
try:
    from google.colab import files
    for fname in ['Q1_rentabilidad_sector.png', 'Q2_top10_rentabilidad.png',
                  'Q3_volumen_volatilidad.png', 'Q4_volumen_trimestre.png',
                  'Q5_dividend_yield.png']:
        files.download(fname)
    print('✅ Gráficas descargadas')
except ImportError:
    print('(No estás en Colab — gráficas guardadas localmente)')

In [ ]:
spark.stop()
print('✅ SparkSession cerrada. Análisis PySpark completado.')